# Testing Context Extraction with local LLM - Codellama:13b

In [268]:
# imports
from urllib import request
import json


In [269]:
# API
url = "http://localhost:11434/api/generate"

## Test Examples

In [270]:
fileCode = """
# Globale Variablen
START = 0
UPPER_BOUND = 100

def sum_n(n):
    return compute_sum(n)


def compute_sum(n):
    total = START
    for i in range(n):
        total += i
    return total


def sum_up_to_limit():
    return compute_sum(UPPER_BOUND)
"""

functionToAnalyze = "func1"

fileCode1 = """
# Globale Variablen
num_a = 10
num_b = 20
c = 30

# Funktionen
def func1():
    return num_a + helper1()

def helper1():
    return num_b + helper2()

def helper2():
    return 42

def dummy1():
    test = 5;
    return test
"""

functionToAnalyze1 = "dummy1"

fileCode2 = """
# Globale Variablen
limit = 10
offset = 3
factor = 2
unused = 999

# Funktionen
def func2(n):
    if n > limit:
        return helperA(n) + offset
    else:
        return helperB(n)

def helperA(x):
    return x * factor

def helperB(x):
    return x + factor

def dummy2():
    return unused
"""

functionToAnalyze2 = "func2"

fileCode3 = """
# Globale Variablen
threshold = 100
step = 3
start = 1

# Funktionen
def func3():
    value = start
    while value < threshold:
        value = helper3(value)
    return value

def helper3(x):
    return x + step

def dummy3():
    return threshold
"""

functionToAnalyze3 = "func3"

fileCode4 = """
# Globale Variablen
base = 0
decrement = 1

# Funktionen
def func4(n):
    if is_base(n):
        return n
    return func4(n - decrement)

def is_base(x):
    return x <= base

def helper_unused():
    return decrement
"""

functionToAnalyze4 = "func4"

fileCode5 = """
# Globale Variablen
MAX = 100
MIN = 1
STEP = 5
factor = 2
unused = 999

# Funktionen
def func5(n):
    total = 0
    for i in range(MIN, n, STEP):
        if i % 2 == 0:
            total += helper_even(i)
        else:
            total += helper_odd(i)
    while total < MAX:
        total = helper_increment(total)
    return total

def helper_even(x):
    return x * factor + helper_increment(x)

def helper_odd(x):
    if x % 3 == 0:
        return x // 2
    else:
        return x + factor

def helper_increment(value):
    if value >= MAX:
        return value
    return value + STEP

def dummy5():
    temp = 42
    return unused
"""

functionToAnalyze5 = "func5"

fileCode6 = """
# Globale Variablen
LIMIT = 50
OFFSET = 3
factor = 2
unused_global = 999

def ufunc7():
    return "I am not used"

# Funktionen
def func6(x):
    result = 0
    for i in range(x):
        if i % 2 == 0:
            result += even(i)
        elif i % 3 == 0:
            result += helper_multiple_three(i)
        else:
            result += helper_odd(i)
    return result

def even(n):
    return n * factor

def helper_odd(n):
    return n + OFFSET

def helper_multiple_three(n):
    if n > LIMIT:
        return LIMIT
    return n * 2

def dummy6():
    temp = 123
    return unused_global
"""

functionToAnalyze6 = "func6"

fileCode7 = """
# Globale Variablen
BASE = 10
STEP = 2
MAX = 100
factor = 3

# Funktionen
def dummy7():
    return 42  # Nicht genutzt

def func7(n):
    total = 0
    while n > 0:
        total += process_step(n)
        n -= STEP
    return finalize(total)

def process_step(value):
    if value % 5 == 0:
        return helper5(value)
    elif value % 3 == 0:
        return helper3(value)
    else:
        return value + factor

def helper5(v):
    return v // 5

def helper3(v):
    return v // 3

def finalize(result):
    return result % MAX

def unused_helper():
    return 0
"""

functionToAnalyze7 = "func7"

fileCode8 = """
# Globale Variablen
LIMIT = 1000
FACTOR = 3
OFFSET = 7
BASE = 5
unused_global = 999

# Funktionen
def dummy_unused1():
    return 0

def func8(n):
    result = BASE
    for i in range(n):
        if i % 2 == 0:
            result += helper_even(i)
        else:
            result += helper_odd(i)
        if result > LIMIT:
            result = cap_value(result)
    return finalize(result)

def helper_even(x):
    if x > OFFSET:
        return x * FACTOR
    else:
        return x + helper_nested(x)

def helper_odd(y):
    total = y
    while total < LIMIT:
        total += helper_even(total) // 2
        if total % 5 == 0:
            total -= helper_subtract(total)
    return total

def helper_nested(z):
    if z % 3 == 0:
        return z // 3 + helper_extra(z)
    return z + 1

def helper_extra(v):
    if v % 7 == 0:
        return v - 1
    return v + 2

def helper_subtract(val):
    return val - OFFSET

def cap_value(val):
    if val > LIMIT:
        return LIMIT
    return val

def finalize(final_result):
    return final_result % FACTOR

def dummy_unused2():
    temp = 123
    return unused_global
"""

functionToAnalyze8 = "func8"



# Versuch 1

In [271]:
def create_context_v1(full_code, target_fn):
    prompt = f"""You are a static program analysis assistant. You reason over Python source code precisely and deterministically.

    Full source code:
    {full_code}

    Task:
    1. Identify ALL functions that are directly or indirectly called by {target_fn}.
    2. Identify ALL global variables used by those functions.
    3. Extract ONLY:
    - the definitions of those global variables
    - the definitions of those functions
    4. Sort the extracted functions topologically:
    - functions first
    - callers after callees

    Rules:
    - Output valid Python Code.
    - Do NOT include unused functions or globals.
    - Do NOT include comments, explanations, or markdown.
    - Preserve exact Python syntax and indentation.
    """

    payload = {
        "model": "codellama:13b",
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.0,
            "top_p": 1.0
        }
    }

    req = request.Request(url,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"}
    )

    with request.urlopen(req) as response:
        raw = json.loads(response.read())["response"]

    # --- normalize newlines (robust against escaped \\n) ---
    formatted = raw.replace("\\n", "\n").strip()

    return formatted

In [272]:
result = create_context_v1(fileCode, functionToAnalyze)

In [273]:
from IPython.display import display, HTML

display(HTML(f"""
<div style="max-height:400px; overflow:auto; border:1px solid #ccc; padding:10px; white-space:pre;">
{result}
</div>
"""))

# Versuch 2

In [288]:
def find_called_functions_names(full_code, target_fn):
    prompt = f"""You are a deterministic Python code extractor.

    Full source code:
    {full_code}

    Task:
    - Return ONLY a JSON object with this exact schema: "called_functions": ["..."]
    - Include {target_fn}.
    - Include only functions that are directly or indirectly called by {target_fn}.

    Rules:
    - Use function names exactly as defined.
    - Output ONLY JSON. No markdown, no code fences, no extra text.
    """

    payload = {
        "model": "codellama:13b",
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.0,
            "top_p": 1.0
        }
    }

    req = request.Request(url,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"}
    )

    with request.urlopen(req) as response:
        raw = json.loads(response.read())["response"]

    # --- normalize newlines (robust against escaped \\n) ---
    formatted = raw.replace("\\n", "\n").strip()

    return formatted


def find_all_global_variables(full_code):
    prompt = f"""You are a deterministic Python code extractor.

    Full source code:
    {full_code}

    Task:
    - Return ONLY a JSON object with this exact schema: "all_variables": ["..."]
    - Identify all global variables that are defined in the 'Full source code'.

    Rules:
    - Use variable names exactly as defined.
    - Output ONLY JSON. No markdown, no code fences, no extra text.
    """

    payload = {
        "model": "codellama:13b",
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.0,
            "top_p": 1.0
        }
    }

    req = request.Request(url,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"}
    )

    with request.urlopen(req) as response:
        raw = json.loads(response.read())["response"]

    # --- normalize newlines (robust against escaped \\n) ---
    formatted = raw.replace("\\n", "\n").strip()

    return formatted

# - Identify the definitions for the functions from 'Relevant Functions' in 'Full source code' and summarize them into one String.
def summarize_called_functions(full_code, called_functions):
    prompt = f"""You are a deterministic Python code extractor.

    Full source code:
    {full_code}

    Relevant Functions:
    {called_functions}

    Task:
    - Extract ONLY the function definitions whose names are in "Relevant function names"
    - Return them concatenated as plain Python code

    Hard rules:
    - Output ONLY Python code. No markdown. No backticks. No explanations. No comments.
    - Do NOT include any global variables, imports, or any other statements.
    - Include each function exactly once.
    - Preserve the original indentation and line breaks exactly.

    Output format constraint:
    - The first non-whitespace characters of your output MUST be 'def' or 'async def'.
    - If you cannot comply exactly, output ONLY this single word: ERROR
    """

    payload = {
        "model": "codellama:13b",
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.0,
            "top_p": 1.0
        }
    }

    req = request.Request(url,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"}
    )

    with request.urlopen(req) as response:
        raw = json.loads(response.read())["response"]

    # --- normalize newlines (robust against escaped \\n) ---
    formatted = raw.replace("\\n", "\n").strip()

    return formatted

def topologically_sort_functions(called_functions_summary):
    prompt = f"""You are a deterministic Python code extractor.

    Task:
    - You are given a set of Python function definitions:
    {called_functions_summary}

    - Sort these functions in topological order based on their dependencies:
        * Functions that do not call any other functions from this set should appear at the top.
        * Functions that call other functions should appear after the ones they call.
    - Keep the function definitions exactly as they are, preserve indentation and line breaks.

    Rules:
    - Output ONLY Python code. No markdown. No backticks. No explanations. No comments.
    - Preserve the original indentation and line breaks exactly.
    """
    payload = {
        "model": "codellama:13b",
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.0,
            "top_p": 1.0
        }
    }

    from urllib import request
    import json

    req = request.Request(url,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"}
    )

    with request.urlopen(req) as response:
        raw = json.loads(response.read())["response"]

    # --- normalize newlines ---
    formatted = raw.replace("\\n", "\n").strip()

    return formatted


def find_all_variable_dependencies(called_functions_summary, global_variables):
    prompt = f"""You are a deterministic Python code extractor.

    Global Variables:
    {global_variables}

    Full source code:
    {called_functions_summary}

    Task:
    - Look at every function in 'Sourcecode' and identify the variables that are being used, that are also in 'Global Variables'.

    Rules:
    - Output ONLY a JSON array of variable names, e.g. ["var_a", "var_b"]
    - Do NOT include the function names in the JSON array.
    - No explanations, no extra text.
    """

    payload = {
        "model": "codellama:13b",
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.0,
            "top_p": 1.0
        }
    }

    req = request.Request(url,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"}
    )

    with request.urlopen(req) as response:
        raw = json.loads(response.read())["response"]

    # --- normalize newlines (robust against escaped \\n) ---
    formatted = raw.replace("\\n", "\n").strip()

    return formatted

def summarize_called_global_variables(full_code, called_global_variables):
    prompt = f"""You are a deterministic Python code extractor.

    Full source code:
    {full_code}

    Global Variables:
    {called_global_variables}

    Task:
    - Identify the definitions of the variables listed in 'Global Variables' and combine them into one String.
    - Do not include functions.

    Rules:
    - Output valid Python code.
    - Preserve indentation and line breaks.
    - No explanations, no comments, no extra text.
    - Do not add comments, explanations, or language identifiers.
    """
    payload = {
        "model": "codellama:13b",
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0.0, "top_p": 1.0},
    }

    req = request.Request(
        url,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )

    with request.urlopen(req) as response:
        raw = json.loads(response.read())["response"]

    formatted = raw.replace("\\n", "\n").strip()
    return formatted



In [289]:
fullFileCode = fileCode
finalFunctionToAnalyse = functionToAnalyze
called_functions = find_called_functions_names(finalFunctionToAnalyse, fullFileCode)
global_variables = find_all_global_variables(fullFileCode)
called_functions_summary = summarize_called_functions(fullFileCode, called_functions)
called_functions_summary_sorted = topologically_sort_functions(called_functions_summary)
called_global_variables = find_all_variable_dependencies(called_functions_summary, global_variables)
called_global_variables_summary = summarize_called_global_variables(fullFileCode, called_global_variables)
result = called_global_variables_summary + "\n" + "\n" + called_functions_summary_sorted

print("\n" + "="*30)
print("Called Functions")
print(called_functions)

print("\n" + "="*30)
print("All Global Variables")
print(global_variables)

print("\n" + "="*30)
print("Called Functions Summary")
print(called_functions_summary)



Called Functions
{
"called_functions": ["compute_sum", "sum_n"]
}

All Global Variables
{
        "all_variables": ["START", "UPPER_BOUND"]
    }

Called Functions Summary
def compute_sum(n):
    total = START
    for i in range(n):
        total += i
    return total

def sum_n(n):
    return compute_sum(n)


In [290]:
print("\n" + "="*30)
print("Called Functions Summary sorted")
print(called_functions_summary_sorted)

print("\n" + "="*30)
print("Called Global Variables")
print(called_global_variables)

print("\n" + "="*30)
print("Called Global Variables Summary")
print(called_global_variables_summary)


Called Functions Summary sorted
def compute_sum(n):
        total = START
        for i in range(n):
            total += i
        return total

def sum_n(n):
    return compute_sum(n)

Called Global Variables
["START", "UPPER_BOUND"]

Called Global Variables Summary
# Globale Variablen
START = 0
UPPER_BOUND = 100


In [291]:
from IPython.display import display, HTML

display(HTML(f"""
<div style="max-height:400px; overflow:auto; border:1px solid #ccc; padding:10px; white-space:pre;">
{result}
</div>
"""))